# KNN (K-Nearest Neighbors) Algorithm

**What is KNN?**
KNN is a simple machine learning algorithm. It looks at the **K closest points** (neighbors) and uses them to decide the answer for a new point.

**Simple example:** You move to a new house. To guess if the area is safe, you ask your 5 nearest neighbors. If most say "safe", you say "safe". That is KNN with K = 5.

**In this notebook we will:**
1. Load data
2. Look at the data
3. Split data into train and test
4. Scale the data
5. Train the KNN model
6. Check the accuracy
7. Find the best K
8. Predict a new flower

## Step 1: Import libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

## Step 2: Load the data

We use the **Iris dataset**. It has measurements of 3 types of flowers.
- **Features (X):** sepal length, sepal width, petal length, petal width
- **Target (y):** flower type (setosa, versicolor, virginica)

In [ ]:
iris = load_iris()

df = pd.DataFrame(iris.data, columns=iris.feature_names)
df["target"] = iris.target
df["flower"] = df["target"].map(dict(enumerate(iris.target_names)))

df.head()

## Step 3: Look at the data

In [ ]:
print("Shape (rows, columns):", df.shape)
print()
print("Missing values:")
print(df.isnull().sum())
print()
print("Count of each flower:")
print(df["flower"].value_counts())

In [ ]:
df.describe()

In [ ]:
sns.pairplot(df.drop(columns="target"), hue="flower")
plt.show()

## Step 4: Split data into train and test

- **Train data (80%)**: the model learns from this.
- **Test data (20%)**: we use this to check the model on new data.

In [ ]:
X = iris.data
y = iris.target

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("Train size:", X_train.shape)
print("Test size:", X_test.shape)

## Step 5: Scale the data

KNN uses **distance**. If one column has big numbers, it will control the result. Scaling puts all columns on the same level.

Note: we `fit` the scaler on **train data only**, then `transform` both train and test.

In [ ]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

## Step 6: Train the KNN model

`n_neighbors=5` means K = 5.

In [ ]:
knn = KNeighborsClassifier(n_neighbors=5)
knn.fit(X_train_scaled, y_train)

## Step 7: Make predictions and check accuracy

In [ ]:
y_pred = knn.predict(X_test_scaled)

print("Accuracy:", accuracy_score(y_test, y_pred))

In [ ]:
print(classification_report(y_test, y_pred, target_names=iris.target_names))

### Confusion matrix

It shows how many predictions were right and wrong.
- Numbers on the **diagonal** = correct predictions.
- Other numbers = mistakes.

In [ ]:
cm = confusion_matrix(y_test, y_pred)

sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=iris.target_names,
            yticklabels=iris.target_names)
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Confusion Matrix")
plt.show()

## Step 8: Find the best K

Which K is best? Let us try K from 1 to 20 and see the accuracy for each.

In [ ]:
k_values = range(1, 21)
accuracies = []

for k in k_values:
    model = KNeighborsClassifier(n_neighbors=k)
    model.fit(X_train_scaled, y_train)
    pred = model.predict(X_test_scaled)
    accuracies.append(accuracy_score(y_test, pred))

plt.figure(figsize=(8, 5))
plt.plot(k_values, accuracies, marker="o")
plt.xticks(k_values)
plt.xlabel("K value")
plt.ylabel("Accuracy")
plt.title("Accuracy for different K values")
plt.grid(True)
plt.show()

best_k = k_values[int(np.argmax(accuracies))]
print("Best K:", best_k, "with accuracy:", max(accuracies))

**Tip:**
- Small K (like 1) can be noisy and overfit.
- Large K can be too smooth and miss patterns.
- Using an **odd** K helps avoid ties.

## Step 9: Predict a new flower

Let us give the model a new flower: sepal length 5.1, sepal width 3.5, petal length 1.4, petal width 0.2.

In [ ]:
new_flower = np.array([[5.1, 3.5, 1.4, 0.2]])

# Always scale new data with the same scaler
new_flower_scaled = scaler.transform(new_flower)

prediction = knn.predict(new_flower_scaled)
print("Predicted flower:", iris.target_names[prediction[0]])

## Bonus: KNN from scratch (to understand how it works)

Steps:
1. Find the distance from the new point to every training point.
2. Pick the K smallest distances.
3. Take a vote. The most common class wins.

In [ ]:
from collections import Counter

def knn_from_scratch(X_train, y_train, new_point, k=5):
    # 1. Euclidean distance to every training point
    distances = np.sqrt(((X_train - new_point) ** 2).sum(axis=1))

    # 2. Positions of the K nearest points
    nearest_idx = np.argsort(distances)[:k]

    # 3. Vote
    nearest_labels = y_train[nearest_idx]
    return Counter(nearest_labels).most_common(1)[0][0]

my_pred = [knn_from_scratch(X_train_scaled, y_train, point, k=5) for point in X_test_scaled]

print("From-scratch accuracy:", accuracy_score(y_test, my_pred))

## Summary

- KNN predicts using the **K nearest neighbors**.
- It uses **distance**, so we must **scale** the data.
- Choose K by testing different values.
- KNN is simple, but it can be slow on very big data.